# Training and Validation for pKa (Chemprop CLI)

## Notebook Contract
- Uses Chemprop CLI for train/predict workflows.
- Keeps the core training function inside this notebook for transparency and reuse.
- Inputs:
  - `./Datasets/trainable_data/pka_chemprop_train_file.csv`
  - Optional descriptors:
    - `./Datasets/trainable_data/pka_chemprop_feat_file.npz`
    - `./Datasets/trainable_data/pka_chemprop_feat_file.csv`
- Outputs per run (under `./chemprop_model_output/modelN/`):
  - trained model artifacts
  - `predictions/set_preds.csv`
  - `pka_predictions_complete.csv`
  - `run_manifest.json`


## 1. Configuration


In [ ]:
import json
import os
import re
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Run controls
RUN_TRAIN = True
RUN_PREDICT = True
RUN_FIGURES = True

# Core config
PROPERTY_CODE = "pka"
TARGET_COLUMN = "pka_value"
SMILES_COLUMN = "SMILES"
SPLIT_COLUMN = "split"

BASE_OUTPUT_DIR = Path("./chemprop_model_output")
TRAIN_DIR = Path("./Datasets/trainable_data")
TRAIN_FILE = TRAIN_DIR / f"{PROPERTY_CODE}_chemprop_train_file.csv"
PREDICTIONS_FILENAME = f"{PROPERTY_CODE}_predictions_complete.csv"

# Chemprop model/training config
BATCH_SIZE = 32
EPOCHS = 100
DEPTH = 4
DROPOUT = 0.025
RANDOM_SEED = 11

# Optional temperature-descriptor / transfer settings
# Set USE_FEATURES = True when pKa_DataPrep is run in temperature-feature mode.
USE_FEATURES = True
USE_TRANSFER = False
CHECKPOINT_PATH = Path("./Pretrained Model/best-epoch=19-val_loss=0.03.ckpt")
FEAT_NPZ = TRAIN_DIR / f"{PROPERTY_CODE}_chemprop_feat_file.npz"
FEAT_CSV = TRAIN_DIR / f"{PROPERTY_CODE}_chemprop_feat_file.csv"

# Evaluation selection (downstream of training)
# None -> use most recently trained model in BASE_OUTPUT_DIR
# int  -> use a specific run, e.g., 11 means ./chemprop_model_output/model11
SELECT_MODEL_NUMBER = None

# Figure settings
TOP_N_SUBGROUPS = 15
MAX_PER_ROW_SUBGROUPS = 5
PARITY_TRAIN_DENSITY_SCALE = 20.0
MAX_HEAVY_ATOMS_HIST = 40
HEAVY_ATOM_HIST_FIGSIZE = (3.5, 1.4)
BEST_WORST_HETEROATOM_CARBON_RATIO_BINS = 16
BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS = True
BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER = 1.5

# Error diagnostics settings
ERROR_REVIEW_SPLIT = 'test'
TOP_N_BEST_POINTS = 5
TOP_N_WORST_POINTS = 5
TOP_N_REVIEW_SUBGROUP_MOLECULES = 5
SUBGROUP_TTEST_MIN_SELECTED_COUNT = 2
SUBGROUP_TTEST_Q_THRESHOLD = 0.05
OUTLIER_Z_THRESHOLD = 3.0
OUTLIER_IQR_MULTIPLIER = 1.5
EXPORT_ERROR_REVIEW = True

print(f"Train file: {TRAIN_FILE}")
print(f"Feature NPZ: {FEAT_NPZ}")
print(f"Use features: {USE_FEATURES}")
print(f"Output root: {BASE_OUTPUT_DIR}")


## 2. Train/Predict Helpers


In [ ]:
def ensure_cli_available() -> None:
    if shutil.which("chemprop") is None:
        raise EnvironmentError("'chemprop' CLI not found in PATH. Activate your Chemprop environment first.")


def _next_model_dir(base_dir: Path) -> Path:
    base_dir.mkdir(parents=True, exist_ok=True)
    existing = [
        d for d in base_dir.iterdir()
        if d.is_dir() and re.match(r"model\d+", d.name)
    ]
    last_num = max((int(re.search(r"\d+", d.name).group()) for d in existing), default=0)
    return base_dir / f"model{last_num + 1}"


def _latest_model_dir(base_dir: Path) -> Path | None:
    if not base_dir.exists():
        return None
    candidates = [
        d for d in base_dir.iterdir()
        if d.is_dir() and re.match(r"model\d+", d.name)
    ]
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: int(re.search(r"\d+", p.name).group()))[-1]


def _run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None)


def _preflight_checks(
    train_file: Path,
    smiles_col: str,
    target_col: str,
    split_col: str,
    use_features: bool,
    feat_npz: Path,
    feat_csv: Path,
    use_transfer: bool,
    checkpoint_path: Path,
) -> None:
    ensure_cli_available()

    if not train_file.exists():
        raise FileNotFoundError(f"Training file not found: {train_file}")

    train_df = pd.read_csv(train_file)
    required_cols = {smiles_col, target_col, split_col}
    missing = required_cols - set(train_df.columns)
    if missing:
        raise ValueError(f"Training file missing required columns: {sorted(missing)}")

    if use_features:
        if not feat_npz.exists():
            raise FileNotFoundError(f"Descriptor NPZ not found: {feat_npz}")
        if not feat_csv.exists():
            raise FileNotFoundError(f"Descriptor CSV not found: {feat_csv}")

    if use_transfer and not checkpoint_path.exists():
        raise FileNotFoundError(f"Transfer checkpoint not found: {checkpoint_path}")


def train_and_predict(
    property_code: str,
    target_col: str,
    smiles_col: str = "SMILES",
    split_col: str = "split",
    base_dir: Path = Path("chemprop_model_output"),
    train_file: Path = Path("trainable_data/property_chemprop_train_file.csv"),
    use_features: bool = False,
    feat_npz: Path | None = None,
    feat_csv: Path | None = None,
    use_transfer: bool = False,
    checkpoint_path: Path | None = None,
    run_train: bool = True,
    run_predict: bool = True,
    epochs: int = 100,
    batch_size: int = 8,
    depth: int = 1,
    dropout: float = 0.025,
    seed: int = 42,
) -> dict:
    # Train a Chemprop regression model and generate merged prediction outputs.
    # Returns a dictionary with key artifact paths for downstream plotting.

    feat_npz = feat_npz or Path("")
    feat_csv = feat_csv or Path("")
    checkpoint_path = checkpoint_path or Path("")

    _preflight_checks(
        train_file=train_file,
        smiles_col=smiles_col,
        target_col=target_col,
        split_col=split_col,
        use_features=use_features,
        feat_npz=feat_npz,
        feat_csv=feat_csv,
        use_transfer=use_transfer,
        checkpoint_path=checkpoint_path,
    )

    if run_train:
        model_dir = _next_model_dir(base_dir)
        model_dir.mkdir(parents=True, exist_ok=True)
    else:
        latest = _latest_model_dir(base_dir)
        if latest is None:
            raise FileNotFoundError(f"No existing model directories found in {base_dir}")
        model_dir = latest

    print(f"Using model directory: {model_dir}")

    train_cmd = [
        "chemprop", "train",
        "--data-path", str(train_file),
        "--task-type", "regression",
        "--splits-column", split_col,
        "--output-dir", str(model_dir),
        "--batch-size", str(batch_size),
        "--epochs", str(epochs),
        "--depth", str(depth),
        "--dropout", str(dropout),
        "--pytorch-seed", str(seed),
    ]

    if use_features:
        train_cmd.extend(["--descriptors-path", str(feat_npz)])

    if use_transfer:
        train_cmd.extend([
            "--checkpoint", str(checkpoint_path),
            "--freeze-encoder",
            "--frzn-ffn-layers", "1",
        ])

    if run_train:
        _run_command(train_cmd)

    preds_dir = model_dir / "predictions"
    preds_dir.mkdir(parents=True, exist_ok=True)

    preds_file = preds_dir / "set_preds.csv"
    merged_output_file = model_dir / f"{property_code}_predictions_complete.csv"

    predict_cmd = [
        "chemprop", "predict",
        "--test-path", str(train_file),
        "--model-paths", str(model_dir),
        "--preds-path", str(preds_file),
    ]

    if use_features:
        predict_cmd.extend(["--descriptors-path", str(feat_npz)])

    if run_predict:
        _run_command(predict_cmd)

    if not preds_file.exists():
        raise FileNotFoundError(f"Predictions file not found: {preds_file}")

    preds_df = pd.read_csv(preds_file)
    train_df = pd.read_csv(train_file)

    complete_data = {
        smiles_col: train_df[smiles_col],
        f"{property_code}_true": train_df[target_col],
        f"{property_code}_pred": preds_df.iloc[:, 1],
        split_col: train_df[split_col],
    }

    if use_features:
        feat_df = pd.read_csv(feat_csv)
        if "T_C" in feat_df.columns:
            complete_data["T_C"] = feat_df["T_C"]

    complete_df = pd.DataFrame(complete_data)
    complete_df.to_csv(merged_output_file, index=False)

    manifest = {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "property_code": property_code,
        "target_column": target_col,
        "smiles_column": smiles_col,
        "split_column": split_col,
        "train_file": str(train_file),
        "model_dir": str(model_dir),
        "preds_file": str(preds_file),
        "merged_output_file": str(merged_output_file),
        "use_features": use_features,
        "use_transfer": use_transfer,
        "checkpoint_path": str(checkpoint_path) if use_transfer else None,
        "epochs": epochs,
        "batch_size": batch_size,
        "depth": depth,
        "dropout": dropout,
        "seed": seed,
        "train_cmd": train_cmd,
        "predict_cmd": predict_cmd,
    }

    manifest_path = model_dir / "run_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    legacy_output_file = model_dir / PREDICTIONS_FILENAME
    if merged_output_file.name != legacy_output_file.name:
        complete_df.to_csv(legacy_output_file, index=False)

    artifacts = {
        "model_dir": model_dir,
        "preds_file": preds_file,
        "merged_output_file": merged_output_file,
        "legacy_output_file": legacy_output_file,
        "manifest_path": manifest_path,
        "metrics_csv": model_dir / "model_0" / "trainer_logs" / "version_0" / "metrics.csv",
        "best_pt": model_dir / "model_0" / "best.pt",
    }

    print(f"Merged predictions saved to: {merged_output_file}")
    print(f"Run manifest saved to: {manifest_path}")

    return artifacts


## 3. Run Train/Predict


In [ ]:
run_artifacts = None
if RUN_TRAIN or RUN_PREDICT:
    run_artifacts = train_and_predict(
        property_code=PROPERTY_CODE,
        target_col=TARGET_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_col=SPLIT_COLUMN,
        base_dir=BASE_OUTPUT_DIR,
        train_file=TRAIN_FILE,
        use_features=USE_FEATURES,
        feat_npz=FEAT_NPZ,
        feat_csv=FEAT_CSV,
        use_transfer=USE_TRANSFER,
        checkpoint_path=CHECKPOINT_PATH,
        run_train=RUN_TRAIN,
        run_predict=RUN_PREDICT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        depth=DEPTH,
        dropout=DROPOUT,
        seed=RANDOM_SEED,
    )


def build_artifacts_for_model(model_dir: Path, property_code: str) -> dict:
    return {
        'model_dir': model_dir,
        'preds_file': model_dir / 'predictions' / 'set_preds.csv',
        'merged_output_file': model_dir / f'{property_code}_predictions_complete.csv',
        'legacy_output_file': model_dir / PREDICTIONS_FILENAME,
        'manifest_path': model_dir / 'run_manifest.json',
        'metrics_csv': model_dir / 'model_0' / 'trainer_logs' / 'version_0' / 'metrics.csv',
        'best_pt': model_dir / 'model_0' / 'best.pt',
    }


if SELECT_MODEL_NUMBER is None:
    if run_artifacts is not None:
        selected_model_dir = Path(run_artifacts['model_dir'])
    else:
        latest = _latest_model_dir(BASE_OUTPUT_DIR)
        if latest is None:
            raise FileNotFoundError(f'No existing model directories found in {BASE_OUTPUT_DIR}')
        selected_model_dir = latest
    selected_label = f'latest ({selected_model_dir.name})'
else:
    selected_model_dir = BASE_OUTPUT_DIR / f'model{int(SELECT_MODEL_NUMBER)}'
    if not selected_model_dir.exists():
        raise FileNotFoundError(f'Selected model directory not found: {selected_model_dir}')
    selected_label = f'model{int(SELECT_MODEL_NUMBER)}'

artifacts = build_artifacts_for_model(selected_model_dir, PROPERTY_CODE)

run_predictions_path = artifacts['merged_output_file']
if not run_predictions_path.exists() and artifacts['legacy_output_file'].exists():
    run_predictions_path = artifacts['legacy_output_file']

if not run_predictions_path.exists():
    raise FileNotFoundError(
        f'Predictions CSV not found for selected run ({selected_label}). '
        f'Looked for: {artifacts["merged_output_file"]} and {artifacts["legacy_output_file"]}'
    )

print(f'Downstream model selection: {selected_label}')
print('Current run artifacts used downstream:')
for k, v in artifacts.items():
    print(f'- {k}: {v}')


## 4. Evaluation Metrics


In [ ]:
metrics_df = pd.read_csv(run_predictions_path)

true_col = f"{PROPERTY_CODE}_true"
pred_col = f"{PROPERTY_CODE}_pred"

required_cols = {true_col, pred_col, SPLIT_COLUMN}
missing = required_cols - set(metrics_df.columns)
if missing:
    raise ValueError(f"Missing required columns for MAE calculation: {sorted(missing)}")

if 'summarize_target_distribution' in globals():
    target_stats_df = summarize_target_distribution(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
    )
else:
    summary_rows = []
    for split_name, frame in [('overall', metrics_df)] + list(metrics_df.groupby(SPLIT_COLUMN, dropna=False)):
        values = pd.to_numeric(frame[true_col], errors='coerce').dropna()
        if values.empty:
            continue
        summary_rows.append({
            'split': split_name,
            'count': int(values.shape[0]),
            'min': float(values.min()),
            'max': float(values.max()),
            'mean': float(values.mean()),
            'median': float(values.median()),
        })
    target_stats_df = pd.DataFrame(summary_rows)

print('Target distribution summary:')
display(target_stats_df)

for split_name in ['train', 'val', 'test']:
    split_df = metrics_df[metrics_df[SPLIT_COLUMN] == split_name].copy()
    if split_df.empty:
        print(f"{split_name.capitalize()} MAE: split not found in {run_predictions_path}")
        continue
    mae = (split_df[true_col] - split_df[pred_col]).abs().mean()
    print(f"{split_name.capitalize()} MAE: {mae:.4f} (n={len(split_df)})")

temperature_summary_df = None
temperature_split_summary_df = None
if 'T_C' in metrics_df.columns:
    temp_eval_df = metrics_df.copy()
    temp_eval_df['T_C'] = pd.to_numeric(temp_eval_df['T_C'], errors='coerce')
    temp_eval_df = temp_eval_df.dropna(subset=['T_C']).copy()
    if not temp_eval_df.empty:
        temp_eval_df['abs_error'] = (temp_eval_df[true_col] - temp_eval_df[pred_col]).abs()
        temperature_summary_df = (
            temp_eval_df
            .groupby('T_C', as_index=False)
            .agg(
                count=('T_C', 'size'),
                mae=('abs_error', 'mean'),
                median_abs_error=('abs_error', 'median'),
                mean_true=(true_col, 'mean'),
                mean_pred=(pred_col, 'mean'),
            )
            .sort_values('T_C')
            .reset_index(drop=True)
        )
        temperature_split_summary_df = (
            temp_eval_df
            .groupby(['split', 'T_C'], as_index=False)
            .agg(
                count=('T_C', 'size'),
                mae=('abs_error', 'mean'),
            )
            .sort_values(['split', 'T_C'])
            .reset_index(drop=True)
        )

        print('\nTemperature distribution and prediction quality:')
        display(temperature_summary_df)
        print('\nTemperature-by-split summary:')
        display(temperature_split_summary_df)


## 5. Figure Helpers


In [ ]:
import importlib
import sys
EXTERNAL_FUNCTIONS_DIR = (Path.cwd().parent / 'External Functions').resolve()
PLOTTING_FUNCTIONS_DIR = (EXTERNAL_FUNCTIONS_DIR / 'Plotting Functions').resolve()
for path in (EXTERNAL_FUNCTIONS_DIR, PLOTTING_FUNCTIONS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import train_prediction_plotting as _train_prediction_plotting
importlib.reload(_train_prediction_plotting)

from train_prediction_plotting import (
    build_prediction_error_table,
    compare_ranked_subgroups_to_training_for_run,
    plot_atom_count_hist,
    plot_best_worst_heteroatom_carbon_ratio_for_run,
    plot_best_worst_molecules_for_run,
    plot_heteroatom_pie,
    plot_loss_curves,
    plot_parity_scatter_for_run,
    plot_parity_scatter_grid_for_run,
    plot_ranked_subgroups_for_run,
    plot_subgroups_for_run,
    plot_train_best_worst_heteroatom_carbon_ratio_for_run,
    plot_training_density_for_run,
    summarize_target_distribution,
)


## 6. Generate Figures


In [ ]:
if RUN_FIGURES:
    figures_dir = artifacts['model_dir'] / 'figures'
    figures_dir.mkdir(parents=True, exist_ok=True)

    plot_loss_curves(artifacts['metrics_csv'], save_path=figures_dir / 'loss_curves.png')

    plot_training_density_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        density_split='train',
        density_scale=PARITY_TRAIN_DENSITY_SCALE,
        save_path=figures_dir / f'{PROPERTY_CODE}_train_density.png',
    )

    plot_parity_scatter_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        split_name='test',
        density_split='train',
        save_path=figures_dir / 'parity_plot_test.png',
    )

    plot_parity_scatter_grid_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        split_names=('train', 'val', 'test'),
        density_split='train',
        save_path=figures_dir / 'parity_plots_all_splits.png',
    )

    plot_subgroups_for_run(
        run_predictions_path,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name='train',
        top_n=TOP_N_SUBGROUPS,
        max_per_row=MAX_PER_ROW_SUBGROUPS,
        title='Top Functional Groups in Training Data',
        outfile=figures_dir / 'top_subgroups_train.png',
    )

    plot_best_worst_molecules_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        top_n=TOP_N_BEST_POINTS,
        bottom_n=TOP_N_WORST_POINTS,
        save_dir=figures_dir,
    )

    plot_ranked_subgroups_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        rank_kind='best',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        top_n_subgroups=TOP_N_SUBGROUPS,
        max_per_row=MAX_PER_ROW_SUBGROUPS,
        title=f'Top Functional Groups in Best {TOP_N_REVIEW_SUBGROUP_MOLECULES} {ERROR_REVIEW_SPLIT.title()} Predictions',
        outfile=figures_dir / f'best_{TOP_N_REVIEW_SUBGROUP_MOLECULES}_{ERROR_REVIEW_SPLIT}_subgroups.png',
    )
    plot_ranked_subgroups_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        rank_kind='worst',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        top_n_subgroups=TOP_N_SUBGROUPS,
        max_per_row=MAX_PER_ROW_SUBGROUPS,
        title=f'Top Functional Groups in Worst {TOP_N_REVIEW_SUBGROUP_MOLECULES} {ERROR_REVIEW_SPLIT.title()} Predictions',
        outfile=figures_dir / f'worst_{TOP_N_REVIEW_SUBGROUP_MOLECULES}_{ERROR_REVIEW_SPLIT}_subgroups.png',
    )

    best_subgroup_stats_df = compare_ranked_subgroups_to_training_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        rank_kind='best',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        min_selected_count=SUBGROUP_TTEST_MIN_SELECTED_COUNT,
    )
    best_subgroup_stats_df.to_csv(
        figures_dir / f'best_{TOP_N_REVIEW_SUBGROUP_MOLECULES}_{ERROR_REVIEW_SPLIT}_subgroup_stats_vs_train.csv',
        index=False,
    )
    print(f'Best subgroup enrichment vs train (q <= {SUBGROUP_TTEST_Q_THRESHOLD}):')
    display(best_subgroup_stats_df[best_subgroup_stats_df['q_value'] <= SUBGROUP_TTEST_Q_THRESHOLD])

    worst_subgroup_stats_df = compare_ranked_subgroups_to_training_for_run(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        rank_kind='worst',
        top_n_molecules=TOP_N_REVIEW_SUBGROUP_MOLECULES,
        min_selected_count=SUBGROUP_TTEST_MIN_SELECTED_COUNT,
    )
    worst_subgroup_stats_df.to_csv(
        figures_dir / f'worst_{TOP_N_REVIEW_SUBGROUP_MOLECULES}_{ERROR_REVIEW_SPLIT}_subgroup_stats_vs_train.csv',
        index=False,
    )
    print(f'Worst subgroup enrichment vs train (q <= {SUBGROUP_TTEST_Q_THRESHOLD}):')
    display(worst_subgroup_stats_df[worst_subgroup_stats_df['q_value'] <= SUBGROUP_TTEST_Q_THRESHOLD])

    plot_atom_count_hist(
        run_predictions_path,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        max_atoms=MAX_HEAVY_ATOMS_HIST,
        hist_bins=32,
        figsize=HEAVY_ATOM_HIST_FIGSIZE,
        save_path=figures_dir / 'atom_count_hist.png',
    )
    plot_heteroatom_pie(
        run_predictions_path,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        save_path=figures_dir / 'heteroatom_pie.png',
    )
    best_worst_heteroatom_carbon_ratio_df = plot_best_worst_heteroatom_carbon_ratio_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        top_n=TOP_N_BEST_POINTS,
        bottom_n=TOP_N_WORST_POINTS,
        bins=BEST_WORST_HETEROATOM_CARBON_RATIO_BINS,
        remove_outliers=BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS,
        outlier_iqr_multiplier=BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER,
        save_path=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_heteroatom_carbon_ratio.png',
    )
    best_worst_heteroatom_carbon_ratio_summary_df = (
        best_worst_heteroatom_carbon_ratio_df
        .dropna(subset=['heteroatom_to_carbon_ratio'])
        .groupby('rank_group', dropna=False)
        .agg(
            n_molecules=('heteroatom_to_carbon_ratio', 'count'),
            n_plotted=('ratio_plotted_in_histogram', 'sum'),
            n_hidden_outliers=('ratio_outlier_for_histogram', 'sum'),
            min=('heteroatom_to_carbon_ratio', 'min'),
            max=('heteroatom_to_carbon_ratio', 'max'),
            mean=('heteroatom_to_carbon_ratio', 'mean'),
            median=('heteroatom_to_carbon_ratio', 'median'),
        )
        .reset_index()
    )
    display(best_worst_heteroatom_carbon_ratio_summary_df)
    best_worst_heteroatom_carbon_ratio_df.to_csv(
        figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_heteroatom_carbon_ratio.csv',
        index=False,
    )
    train_best_worst_heteroatom_carbon_ratio_df = plot_train_best_worst_heteroatom_carbon_ratio_for_run(
        run_predictions_path,
        property_col=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_name=ERROR_REVIEW_SPLIT,
        train_split_name='train',
        top_n=TOP_N_BEST_POINTS,
        bottom_n=TOP_N_WORST_POINTS,
        bins=BEST_WORST_HETEROATOM_CARBON_RATIO_BINS,
        remove_outliers=BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS,
        outlier_iqr_multiplier=BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER,
        save_path=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_train_best_worst_heteroatom_carbon_ratio.png',
    )
    train_best_worst_heteroatom_carbon_ratio_summary_df = (
        train_best_worst_heteroatom_carbon_ratio_df
        .dropna(subset=['heteroatom_to_carbon_ratio'])
        .groupby('rank_group', dropna=False)
        .agg(
            n_molecules=('heteroatom_to_carbon_ratio', 'count'),
            n_plotted=('ratio_plotted_in_histogram', 'sum'),
            n_hidden_outliers=('ratio_outlier_for_histogram', 'sum'),
            min=('heteroatom_to_carbon_ratio', 'min'),
            max=('heteroatom_to_carbon_ratio', 'max'),
            mean=('heteroatom_to_carbon_ratio', 'mean'),
            median=('heteroatom_to_carbon_ratio', 'median'),
        )
        .reset_index()
    )
    display(train_best_worst_heteroatom_carbon_ratio_summary_df)
    train_best_worst_heteroatom_carbon_ratio_df.to_csv(
        figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_train_best_worst_heteroatom_carbon_ratio.csv',
        index=False,
    )

    if temperature_summary_df is not None and not temperature_summary_df.empty:
        temp_plot_df = temperature_summary_df.copy()
        fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
        axes[0].bar(temp_plot_df['T_C'], temp_plot_df['count'], color='#4C72B0', width=0.8)
        axes[0].set_ylabel('Count')
        axes[0].set_title('pKa temperature distribution and prediction error')

        axes[1].plot(temp_plot_df['T_C'], temp_plot_df['mae'], color='#DD8452', marker='o', linewidth=1.8)
        axes[1].set_ylabel('MAE')
        axes[1].set_xlabel('Temperature (C)')

        fig.tight_layout()
        temp_plot_path = figures_dir / 'temperature_distribution_and_mae.png'
        fig.savefig(temp_plot_path, dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f'Temperature summary figure saved to: {temp_plot_path}')

    print(f'Figures saved to: {figures_dir}')
else:
    print('RUN_FIGURES is False; skipping figure generation.')


## 7. Error Diagnostics


In [ ]:
error_review_df = build_prediction_error_table(
    run_predictions_path,
    property_code=PROPERTY_CODE,
    split_col=SPLIT_COLUMN,
    smiles_col=SMILES_COLUMN,
    split_name=ERROR_REVIEW_SPLIT,
    outlier_z_threshold=OUTLIER_Z_THRESHOLD,
    outlier_iqr_multiplier=OUTLIER_IQR_MULTIPLIER,
)

best_points_df = error_review_df.sort_values('abs_error', ascending=True).head(TOP_N_BEST_POINTS).copy()
worst_points_df = error_review_df.head(TOP_N_WORST_POINTS).copy()
outlier_df = error_review_df[error_review_df['flag_potential_outlier']].copy()

display_cols = [
    'error_rank', SMILES_COLUMN, true_col, pred_col,
    'residual', 'abs_error', 'residual_z',
    'flag_large_abs_error', 'flag_large_residual_z', 'flag_potential_outlier'
]

print(f"Error diagnostics split: {ERROR_REVIEW_SPLIT} (n={len(error_review_df)})")
print(f"IQR threshold (abs error): {error_review_df['abs_error_iqr_threshold'].iloc[0]:.4f}")
print(f"Potential outliers flagged: {len(outlier_df)}")

print(f"\nTop {min(TOP_N_BEST_POINTS, len(error_review_df))} best predictions:")
display(best_points_df[display_cols])

print(f"\nTop {min(TOP_N_WORST_POINTS, len(error_review_df))} worst predictions:")
display(worst_points_df[display_cols])

if len(outlier_df) > 0:
    print('\nFlagged potential outliers:')
    display(outlier_df[display_cols])
else:
    print('\nNo potential outliers flagged with current thresholds.')

if EXPORT_ERROR_REVIEW:
    review_out_path = Path(artifacts['model_dir']) / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_error_review.csv'
    error_review_df.to_csv(review_out_path, index=False)
    print(f'\nSaved full error review table to: {review_out_path}')


## 8. Run Summary


In [ ]:
print("Run summary:")
print(f"- Property code: {PROPERTY_CODE}")
print(f"- Train file: {TRAIN_FILE}")
print(f"- Model directory: {artifacts['model_dir']}")
print(f"- Best model: {artifacts['best_pt']}")
print(f"- Metrics CSV: {artifacts['metrics_csv']}")
print(f"- Predictions CSV: {run_predictions_path}")
print(f"- Manifest: {artifacts['manifest_path']}")
